In [1]:
"""
check_inclusion_exclusion_balance.py

Two things this checks, directly on your real data:

1. What's the actual inclusion vs exclusion split, overall and per-trial?
   Also flags trials with ZERO exclusion criteria detected at all -- that's
   the strongest signal of a missed "Exclusion Criteria:" header, since real
   trials almost always have at least one exclusion rule (age, pregnancy,
   allergy, prior participation, etc. are near-universal).

2. Are there criteria tagged is_inclusion=True whose TEXT sounds like an
   exclusion rule (contains "must not", "no history of", "without", etc)?
   These are candidate mis-tags from a missed header.

Run this on your real structured_clinical_trials.json before deciding
whether/how to fix header detection.
"""
import json

TRIALS_PATH = "processed_data/10000_trials/structured_clinical_trials.json"

# Phrases that strongly suggest "this sentence is functionally an exclusion
# rule" regardless of which section it landed in.
NEGATION_CUES = [
    "must not", "no history of", "without", "absence of", "excluded if",
    "not eligible", "cannot have", "contraindicat", "not permitted",
    "no prior", "no known", "not allowed", "must be free of", "no evidence of",
]


def sounds_like_exclusion(text: str) -> bool:
    lower = text.lower()
    return any(cue in lower for cue in NEGATION_CUES)


def main():
    with open(TRIALS_PATH) as f:
        trials = json.load(f)

    total_inc, total_exc = 0, 0
    trials_with_zero_exc = 0
    trials_with_zero_inc = 0
    suspicious_mistags = []

    for t in trials:
        criteria = t.get("criteria", [])
        inc_count = sum(1 for c in criteria if c.get("is_inclusion"))
        exc_count = sum(1 for c in criteria if not c.get("is_inclusion"))
        total_inc += inc_count
        total_exc += exc_count

        if exc_count == 0 and len(criteria) > 0:
            trials_with_zero_exc += 1
        if inc_count == 0 and len(criteria) > 0:
            trials_with_zero_inc += 1

        for c in criteria:
            if c.get("is_inclusion") and sounds_like_exclusion(c.get("raw_entity", "")):
                suspicious_mistags.append((t.get("nct_id", "?"), c["raw_entity"][:100]))

    total = total_inc + total_exc
    print("=" * 60)
    print("INCLUSION vs EXCLUSION BALANCE")
    print("=" * 60)
    print(f"Total criteria: {total}")
    print(f"  Inclusion: {total_inc} ({total_inc/total*100:.1f}%)")
    print(f"  Exclusion: {total_exc} ({total_exc/total*100:.1f}%)")
    print(f"\nTrials with ZERO exclusion criteria detected: {trials_with_zero_exc} / {len(trials)} "
          f"({trials_with_zero_exc/len(trials)*100:.1f}%)")
    print(f"Trials with ZERO inclusion criteria detected: {trials_with_zero_inc} / {len(trials)} "
          f"({trials_with_zero_inc/len(trials)*100:.1f}%)")

    print(f"\n{'='*60}")
    print(f"CRITERIA TAGGED 'INCLUSION' THAT SOUND LIKE EXCLUSION RULES")
    print(f"(likely mis-tags from a missed section header)")
    print("=" * 60)
    print(f"Found {len(suspicious_mistags)} suspicious lines")
    for nct_id, text in suspicious_mistags[:15]:
        print(f"  [{nct_id}] {text}")

    if trials_with_zero_exc / len(trials) > 0.15:
        print("\n⚠ Over 15% of trials show ZERO exclusion criteria. Real clinical trials "
              "almost never have none -- this strongly suggests header detection is "
              "missing non-standard exclusion headers on a meaningful chunk of your data.")


if __name__ == "__main__":
    main()

INCLUSION vs EXCLUSION BALANCE
Total criteria: 7591
  Inclusion: 3111 (41.0%)
  Exclusion: 4480 (59.0%)

Trials with ZERO exclusion criteria detected: 11 / 550 (2.0%)
Trials with ZERO inclusion criteria detected: 1 / 550 (0.2%)

CRITERIA TAGGED 'INCLUSION' THAT SOUND LIKE EXCLUSION RULES
(likely mis-tags from a missed section header)
Found 89 suspicious lines
  [NCT07735182] 3. No use of antibiotics or probiotics, and no history of acute infection within 1 month prior to en
  [NCT07734831] * Diagnosis of coronary artery disease (CAD), including patients with or without acute coronary synd
  [NCT07734831] * Clinically stable patients without limiting angina.
  [NCT07729995] 3. Current or prior malignancy, unless the malignancy was treated with curative intent and the parti
  [NCT07729410] * No cardiovascular contraindications to exercise testing or participation in exercise-based cardiac
  [NCT07726589] * Chronic kidney disease (CKD) stages 1-5, classified by estimated glomerular filtra